# Reliability checks for the citrus benchmark (run AFTER the main notebook)
The main run gave 99.7–100 % accuracy. Before publishing, we test whether the models
could be using cues other than disease symptoms. Four checks:

1. **Acquisition differences** – file format, resolution, background colour per class
2. **Background-only test** – can a simple classifier tell the classes apart from 8 border pixels? (Noyan 2022)
3. **Hidden duplicates** – rotated / flipped / shifted copies that the first duplicate check cannot see;
   then re-run 5-fold CV with duplicate groups kept in the same fold (no leakage)
4. **Shortcut stress test** – re-encode every test image as JPEG and replace the background with flat grey,
   then measure the accuracy of the already-trained fold-1 models

Every result is saved to `OUT_DIR` with the prefix `check_`. Runtime: T4 GPU, ~45–60 min.

In [ ]:
!pip install -q imagehash
import os, io, copy, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix

from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive/Citrus_Plant_Leaves")
OUT_DIR  = os.environ.get("OUT_DIR",  "/content/drive/MyDrive/citrus_results")   # same folder as the main run
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"
SIM_THRESHOLD = 0.95          # cosine similarity above which two images are treated as copies of one leaf
GROUP_MODELS = ["shufflenet_v2", "resnet18"]
EPOCHS, PATIENCE, LR, BATCH, IMG = (2, 1, 1e-4, 16, 224) if QUICK_TEST else (20, 5, 1e-4, 32, 224)
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

## 0. Rebuild the exact image list used in the main run

In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
class_dirs = {}
for d, _, files in os.walk(DATA_DIR):
    if any(f.lower().endswith(EXTS) for f in files): class_dirs[os.path.basename(d)] = d
CLASSES = sorted(class_dirs, key=lambda c: (0 if c.lower().startswith("healthy") else 1, c))
removed = set(pd.read_csv(f"{OUT_DIR}/duplicates_removed.csv")["removed"]) if os.path.exists(f"{OUT_DIR}/duplicates_removed.csv") else set()
removed_names = {os.path.basename(p) + "|" + os.path.basename(os.path.dirname(p)) for p in removed}
paths, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(class_dirs[c])):
        if f.lower().endswith(EXTS) and f"{f}|{c}" not in removed_names:
            paths.append(os.path.join(class_dirs[c], f)); labels.append(i)
labels = np.array(labels)
if QUICK_TEST:
    idx = np.concatenate([np.where(labels == k)[0][:12] for k in (0, 1)])
    paths, labels = [paths[i] for i in idx], labels[idx]
print(len(paths), "images;", dict(zip(CLASSES, np.bincount(labels))))

## 1. Acquisition differences between classes (format, size, background colour)

In [ ]:
def border_pixels(im, k=8):
    """8 pixels: 4 corners + 4 edge mid-points, 3 px in from the edge (Noyan 2022)."""
    a = np.asarray(im.convert("RGB").resize((256, 256)), dtype=np.float32)
    pts = [(3,3),(3,128),(3,252),(128,3),(128,252),(252,3),(252,128),(252,252)]
    return np.concatenate([a[r, c] for r, c in pts])

rows, bg = [], []
for p, y in zip(paths, labels):
    with Image.open(p) as im:
        rows.append({"class": CLASSES[y], "format": (im.format or os.path.splitext(p)[1][1:]).upper(),
                     "mode": im.mode, "width": im.size[0], "height": im.size[1]})
        bg.append(border_pixels(im))
meta = pd.DataFrame(rows); bg = np.array(bg)
fmt = pd.crosstab(meta["class"], meta["format"])
size = meta.groupby("class")[["width", "height"]].agg(["min", "median", "max"])
bg_rgb = pd.DataFrame(bg.reshape(len(bg), 8, 3).mean(1), columns=["R", "G", "B"]); bg_rgb["class"] = meta["class"]
bg_mean = bg_rgb.groupby("class").mean().round(1)
print("File format by class:\n", fmt, "\n\nImage size by class:\n", size, "\n\nMean background colour (RGB):\n", bg_mean)
fmt.to_csv(f"{OUT_DIR}/check1_format_by_class.csv"); size.to_csv(f"{OUT_DIR}/check1_size_by_class.csv")
bg_mean.to_csv(f"{OUT_DIR}/check1_background_rgb.csv")

## 2. Background-only classifier (8 border pixels → Random Forest, 5-fold CV)
Disease information is NOT in these pixels. Accuracy far above 50 % = the classes differ in how they were photographed.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=SEED)
cv = StratifiedKFold(5 if not QUICK_TEST else 2, shuffle=True, random_state=SEED)
bg_acc = cross_val_score(rf, bg, labels, cv=cv, scoring="accuracy")
majority = np.bincount(labels).max() / len(labels)
print(f"Background-only accuracy: {bg_acc.mean()*100:.2f} ± {bg_acc.std()*100:.2f} %  (majority-class baseline {majority*100:.2f} %)")
json.dump({"bg_only_acc_mean": float(bg_acc.mean()), "bg_only_acc_sd": float(bg_acc.std()),
           "majority_baseline": float(majority)}, open(f"{OUT_DIR}/check2_background_only.json", "w"), indent=2)

## 3. Hidden duplicates: rotated / flipped / shifted copies
Each image is compared with all 8 rotations/flips of every other image using ImageNet ResNet-18 features.
Pairs above `SIM_THRESHOLD` are joined into "leaf groups". **Look at the pair figure** to confirm they are real copies.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
emb_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
fe = models.resnet18(weights="DEFAULT" if PRETRAINED else None); fe.fc = nn.Identity(); fe.eval().to(DEVICE)

def dihedral(im):
    out = []
    for flip in (False, True):
        b = ImageOps.mirror(im) if flip else im
        for r in (0, 90, 180, 270): out.append(b.rotate(r, expand=True))
    return out

feats = []          # shape N x 8 x 512
with torch.no_grad():
    for p in paths:
        im = Image.open(p).convert("RGB")
        x = torch.stack([emb_tf(v) for v in dihedral(im)]).to(DEVICE)
        f = nn.functional.normalize(fe(x), dim=1).cpu().numpy(); feats.append(f)
feats = np.stack(feats)
base = feats[:, 0, :]                                        # original orientation
sim = np.max(np.einsum("id,jkd->ijk", base, feats), axis=2)  # best match over the 8 variants of j
sim = np.maximum(sim, sim.T); np.fill_diagonal(sim, 0)

i_idx, j_idx = np.where(np.triu(sim) >= SIM_THRESHOLD)
pairs = pd.DataFrame({"img_a": [paths[i] for i in i_idx], "img_b": [paths[j] for j in j_idx],
                      "similarity": sim[i_idx, j_idx], "cross_class": labels[i_idx] != labels[j_idx]})
pairs.sort_values("similarity", ascending=False).to_csv(f"{OUT_DIR}/check3_hidden_duplicate_pairs.csv", index=False)
n = len(paths)
def make_groups(thr):
    a, b = np.where(np.triu(sim) >= thr)
    keep = labels[a] == labels[b]                      # only same-class copies form a leaf group
    adj = coo_matrix((np.ones(keep.sum()), (a[keep], b[keep])), shape=(n, n))
    return connected_components(adj, directed=False)
thr = SIM_THRESHOLD
n_groups, groups = make_groups(thr)
# safety: one giant group means the threshold is too loose -> tighten it automatically
while np.bincount(groups).max() > max(5, 0.05 * n) and thr < 0.995:
    thr = round(thr + 0.01, 3); n_groups, groups = make_groups(thr)
if thr != SIM_THRESHOLD: print(f"Threshold raised to {thr} to avoid one giant group – report this value in the paper.")
gsize = np.bincount(groups)
print(f"Pairs above {SIM_THRESHOLD}: {len(pairs)} ({pairs.cross_class.sum()} cross-class)")
print(f"Leaf groups: {n_groups} from {n} images; images in a group of size >1: {(gsize[groups] > 1).sum()}")
json.dump({"threshold_used": thr, "pairs_at_input_threshold": int(len(pairs)),
           "cross_class_pairs": int(pairs.cross_class.sum()), "n_groups": int(n_groups),
           "images_in_multi_image_groups": int((gsize[groups] > 1).sum()), "largest_group": int(gsize.max())},
          open(f"{OUT_DIR}/check3_duplicate_summary.json", "w"), indent=2)

# similarity distribution figure + example pairs for visual confirmation
plt.figure(figsize=(6, 3.5)); plt.hist(sim.max(1), bins=50); plt.axvline(SIM_THRESHOLD, color="r", ls="--")
plt.xlabel("Highest similarity to any other image"); plt.ylabel("Images"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/check3_similarity_hist.png", dpi=300); plt.show()
show = pairs.sort_values("similarity").head(6)       # weakest pairs above threshold = hardest to confirm
if len(show):
    fig, ax = plt.subplots(2, len(show), figsize=(2.6 * len(show), 5.4), squeeze=False)
    for c, (_, r) in enumerate(show.iterrows()):
        for rr, key in enumerate(("img_a", "img_b")):
            ax[rr, c].imshow(Image.open(r[key]).convert("RGB")); ax[rr, c].axis("off")
        ax[0, c].set_title(f"sim {r.similarity:.3f}", fontsize=9)
    plt.suptitle("Pairs just above the threshold – are these the same leaf?"); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/check3_example_pairs.png", dpi=200); plt.show()

## 3b. Leakage-free CV: duplicate groups kept in the same fold (StratifiedGroupKFold)

In [ ]:
train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = emb_tf

class DS(Dataset):
    def __init__(s, items, y, tf): s.items, s.y, s.tf = items, y, tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        im = s.items[i] if isinstance(s.items[i], Image.Image) else Image.open(s.items[i]).convert("RGB")
        return s.tf(im), int(s.y[i])

def build_model(name):
    W = "DEFAULT" if PRETRAINED else None
    if name == "shufflenet_v2": m = models.shufflenet_v2_x1_0(weights=W); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "resnet18":    m = models.resnet18(weights=W); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "mobilenet_v2": m = models.mobilenet_v2(weights=W); m.classifier[1] = nn.Linear(m.last_channel, 2)
    elif name == "efficientnet_b0": m = models.efficientnet_b0(weights=W); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif name == "squeezenet1_1": m = models.squeezenet1_1(weights=W); m.classifier[1] = nn.Conv2d(512, 2, 1)
    return m

def predict(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.append(torch.softmax(model(x.to(DEVICE)), 1)[:, 1].cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def train_eval(name, tr, va):
    set_seed()
    tr_dl = DataLoader(DS([paths[i] for i in tr], labels[tr], train_tf), BATCH, shuffle=True, num_workers=0 if QUICK_TEST else 2)
    va_dl = DataLoader(DS([paths[i] for i in va], labels[va], eval_tf), BATCH, num_workers=0 if QUICK_TEST else 2)
    m = build_model(name).to(DEVICE)
    w = torch.tensor(len(tr) / (2 * np.bincount(labels[tr], minlength=2)), dtype=torch.float32, device=DEVICE)
    crit, opt = nn.CrossEntropyLoss(weight=w), torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    best, state, wait = 1e9, None, 0
    for ep in range(EPOCHS):
        m.train()
        for x, y in tr_dl:
            opt.zero_grad(); loss = crit(m(x.to(DEVICE)), y.to(DEVICE)); loss.backward(); opt.step()
        m.eval(); vl = 0
        with torch.no_grad():
            for x, y in va_dl: vl += crit(m(x.to(DEVICE)), y.to(DEVICE)).item() * len(y)
        vl /= len(va)
        if vl < best: best, state, wait = vl, copy.deepcopy(m.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE: break
    m.load_state_dict(state); y, p = predict(m, va_dl); yhat = (p >= .5).astype(int)
    return {"accuracy": accuracy_score(y, yhat), "f1": f1_score(y, yhat, zero_division=0),
            "auc": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan}

sgkf = StratifiedGroupKFold(n_splits=5 if not QUICK_TEST else 2, shuffle=True, random_state=SEED)
grp_rows = []
for name in GROUP_MODELS:
    for k, (tr, va) in enumerate(sgkf.split(paths, labels, groups)):
        r = train_eval(name, tr, va); r.update({"model": name, "fold": k + 1}); grp_rows.append(r)
        print(f"{name:14s} group-fold {k+1}: acc={r['accuracy']:.4f} f1={r['f1']:.4f}")
grp = pd.DataFrame(grp_rows); grp.to_csv(f"{OUT_DIR}/check3b_groupcv_per_fold.csv", index=False)
grp_sum = grp.groupby("model")[["accuracy", "f1", "auc"]].agg(["mean", "std"]); print(grp_sum)
grp_sum.to_csv(f"{OUT_DIR}/check3b_groupcv_summary.csv")

## 4. Shortcut stress test on the fold-1 models from the main run
Same validation images (fold 1 of the main run), four versions:
original · all re-encoded as JPEG (q=90) · background replaced by flat grey · both.
A large accuracy drop = the model relied on format/background, not on the leaf.

In [ ]:
def leaf_mask(im):
    """Leaf = pixels clearly more saturated than the pale background (Otsu on HSV saturation)."""
    s = np.asarray(im.convert("HSV"))[:, :, 1].astype(np.uint8)
    hist = np.bincount(s.ravel(), minlength=256).astype(float); w = np.cumsum(hist); mu = np.cumsum(hist * np.arange(256))
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu[-1] * w / w[-1] - mu) ** 2 / (w * (w[-1] - w))
    t = int(np.nanargmax(between[:-1])); return s > t

def to_jpeg(im, q=90):
    b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=q); b.seek(0); return Image.open(b).convert("RGB")

def grey_bg(im):
    a = np.asarray(im.convert("RGB")).copy(); a[~leaf_mask(im)] = (128, 128, 128); return Image.fromarray(a)

VARIANTS = {"original": lambda im: im, "jpeg_q90": to_jpeg, "grey_background": grey_bg,
            "jpeg_plus_grey": lambda im: grey_bg(to_jpeg(im))}
skf = StratifiedKFold(n_splits=5 if not QUICK_TEST else 2, shuffle=True, random_state=SEED)
va = list(skf.split(paths, labels))[0][1]            # identical to main-run fold 1
base_imgs = [Image.open(paths[i]).convert("RGB") for i in va]

stress = []
for name in ["mobilenet_v2", "efficientnet_b0", "shufflenet_v2", "resnet18", "squeezenet1_1"]:
    ck = f"{OUT_DIR}/{name}_fold1.pt"
    if not os.path.exists(ck): continue
    m = build_model(name); m.load_state_dict(torch.load(ck, map_location="cpu")); m.to(DEVICE)
    row = {"model": name}
    for vname, fn in VARIANTS.items():
        dl = DataLoader(DS([fn(im) for im in base_imgs], labels[va], eval_tf), BATCH)
        y, p = predict(m, dl); row[vname] = accuracy_score(y, (p >= .5).astype(int)) * 100
    stress.append(row); print(row)
stress = pd.DataFrame(stress).round(2); stress.to_csv(f"{OUT_DIR}/check4_shortcut_stress.csv", index=False)

# example of the grey-background version (for the paper's supplementary figure)
fig, ax = plt.subplots(2, 4, figsize=(10, 5))
for c, i in enumerate([j for j in range(len(va)) if labels[va][j] == 0][:2] + [j for j in range(len(va)) if labels[va][j] == 1][:2]):
    ax[0, c].imshow(base_imgs[i]); ax[1, c].imshow(grey_bg(base_imgs[i]))
    ax[0, c].set_title(CLASSES[labels[va][i]]); ax[0, c].axis("off"); ax[1, c].axis("off")
plt.suptitle("Original (top) and background replaced by grey (bottom)"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/check4_grey_background_examples.png", dpi=200); plt.show()

print("\nDone. Send these files back:")
for f in sorted(os.listdir(OUT_DIR)):
    if f.startswith("check"): print("  ", f)